# Evaluasi RAG di Redis dengan ragas

Semua langkah memakai `zul.evaluation` (lihat panduan *Mengevaluasi retrieval dan RAG* di dokumentasi Zul). Kunci dan alamat server hanya dari `.env` di akar repo. Jalankan kernel dari folder `framework/studies/retrieval_evaluation`.

In [ ]:
import pandas as pd
from dotenv import load_dotenv

from zul.evaluation.config import RedisSettings
from zul.evaluation.dataset import describe_dataset, load_qa_dataset
from zul.evaluation.generation import ChatGenerator
from zul.evaluation.hits import retrieval_hit_stats
from zul.evaluation.indexing import redis_schema
from zul.evaluation.judge import DEFAULT_METRICS, evaluate_rag
from zul.evaluation.reports import compare_reports, save_report
from zul.evaluation.retrievers import RedisRetriever
from zul.evaluation.runner import length_summary, run_rag
from zul.utilities.embedding_service import AIService

load_dotenv("../../../.env")
ai = AIService.from_env()  # LLM + embedding yang DINILAI
print(ai.config.safe_summary())

## 1. Dataset tanya-jawab

In [ ]:
rows = load_qa_dataset(
    "data/question_answer.xlsx",
    "data/for_benchmarking/rag_evaluation_data2.xlsx",
)
describe_dataset(rows)

## 2. Index di Redis

Index sudah terisi dari `pipline_performance_redis_from_sgd.ipynb`; di sini hanya dipasang.

In [ ]:
db = RedisSettings.from_env().connect()
print(db.check_requirements())

DIMS = 2560  # Qwen3-Embedding-4B
schema_recursive = redis_schema("recursives_redis_sgd", prefix="recursive", dims=DIMS)
schema_pages = redis_schema("pages_redis_sgd", prefix="page", dims=DIMS)

## 3. Jalankan RAG untuk tiap varian

Satu objek retriever per varian; CSV ditulis sambil berjalan dan bisa dilanjutkan bila terputus.

In [ ]:
generator = ChatGenerator(ai)

varian = {
    "redis-recursive-rrf": (schema_recursive, "rrf"),
    "redis-recursive-hybrid": (schema_recursive, "hybrid"),
    "redis-pages-hybrid": (schema_pages, "hybrid"),
}

hasil = {}
for nama, (schema, mode) in varian.items():
    db.create_index(schema, overwrite=False)  # pasang index yang sudah terisi
    retriever = RedisRetriever(db, ai, mode=mode, name=nama)
    hasil[nama] = run_rag(
        rows, retriever, generator, top_k=5, out_path=f"outputs/{nama}.csv"
    )
    print(nama, length_summary(hasil[nama]), retrieval_hit_stats(hasil[nama]))

## 4. Nilai dengan ragas

Penilai dibaca dari `RAGAS_JUDGE_*` di `.env`. Mulai dari metrik retrieval; `faithfulness` dan `answer_relevancy` menyusul bila jawabannya ikut dinilai.

In [ ]:
laporan = {}
for nama, hasil_rag in hasil.items():
    laporan[nama] = evaluate_rag(hasil_rag, metrics=DEFAULT_METRICS, name=nama)
    save_report(laporan[nama], "outputs")
    print(laporan[nama])

In [ ]:
tabel = compare_reports(*[f"outputs/{nama}.scores.json" for nama in varian])
pd.DataFrame(tabel).set_index("name")

## 5. Melihat satu pertanyaan

Untuk memeriksa mengapa skor retrieval rendah: bandingkan konteks yang terambil dengan jawaban acuannya.

In [ ]:
i = 5
baris = hasil["redis-recursive-rrf"][i]
print("pertanyaan:", baris["user_input"])
print("acuan     :", baris["reference"])
for k, c in enumerate(baris["retrieved_contexts"], 1):
    print(f"\n[{k}] {c[:300]}")